# Week 24, Day 2: Markov chains and the noisy ion channel

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w024/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: The stationary distribution

Find `pi` as the eigenvector of P transposed with eigenvalue closest to 1, normalised to sum to 1. Then simulate 200,000 steps of the chain from state 0, choosing each next state by comparing a uniform random number with the cumulative sums of the current row, and store the fraction of time in each state in `frac`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
P = np.array([[0.90, 0.10, 0.00],
              [0.05, 0.85, 0.10],
              [0.00, 0.20, 0.80]])
lam, vecs = np.linalg.eig(P.T)
v = np.real(vecs[:, np.argmin(np.abs(lam - 1))])
pi = ...

r_m = np.random.default_rng(1981)
u = r_m.random(200000); cum = np.cumsum(P, axis=1)
state, counts = 0, np.zeros(3)
for k in range(200000):
    counts[state] += 1
    state = ...                                  # first j with u[k] below cum[state, j]
frac = counts / counts.sum()
print("stationary pi:", pi.round(4), " simulated fractions:", frac.round(4))

In [ ]:
import numpy as np
def test_pi():
    "pi is a probability vector with pi P = pi"
    assert abs(pi.sum() - 1) < 1e-12 and np.all(pi >= 0) and np.abs(pi @ P - pi).max() < 1e-12

def test_sim():
    "the fraction of time spent in each state over the simulation is within 0.02 of pi"
    assert np.abs(frac - pi).max() < 0.02

test_pi()
test_sim()
print("All checks passed")

### Exercise: An exact single-channel record

With alpha = 0.2 /ms and beta = 0.5 /ms, start shut and generate 20,000 dwell times, alternating shut and open, each drawn from an exponential with mean 1/(rate out of the current state). Store the open dwell times in `open_t`, the shut ones in `shut_t`, and the fraction of total time open in `p_open`.

In [ ]:
alpha, beta = 0.2, 0.5
r_g = np.random.default_rng(1977)
open_t, shut_t, is_open = [], [], False
for _ in range(20000):
    rate = beta if is_open else alpha
    d = ...
    (open_t if is_open else shut_t).append(d)
    is_open = not is_open
open_t, shut_t = np.array(open_t), np.array(shut_t)
p_open = ...
print(f"mean open {open_t.mean():.3f} ms, mean shut {shut_t.mean():.3f} ms, fraction open {p_open:.4f} (theory {alpha / (alpha + beta):.4f})")
plt.hist(open_t, 60, density=True, alpha=0.6, label="open"); tt = np.linspace(0, 12, 200); plt.plot(tt, beta * np.exp(-beta * tt))
plt.xlabel("dwell time (ms)"); plt.legend(); plt.show()

In [ ]:
import numpy as np
def test_dwell():
    "mean open time is within 5% of 1/beta = 2 ms and mean shut time within 5% of 1/alpha = 5 ms"
    assert abs(open_t.mean() - 2) / 2 < 0.05 and abs(shut_t.mean() - 5) / 5 < 0.05

def test_popen():
    "the fraction of time open is within 0.02 of alpha / (alpha + beta)"
    assert abs(p_open - 0.2 / 0.7) < 0.02

def test_exponential():
    "open times are exponential: their standard deviation is within 7% of their mean"
    assert abs(open_t.std() / open_t.mean() - 1) < 0.07

test_dwell()
test_popen()
test_exponential()
print("All checks passed")

### Exercise: Channel noise shrinks as one over the square root of N

With alpha = 0.3 /ms and beta = 0.7 /ms (so p = 0.3), simulate N = 10, 100 and 1,000 channels in steps of dt = 0.01 ms: each step, the number of shut channels that open is binomial(n_shut, alpha dt) and the number of open ones that close is binomial(n_open, beta dt). After 2,000 steps of settling, record the fraction open for 100,000 steps and store its standard deviation in `sds[N]`.

In [ ]:
a2, b2, dt = 0.3, 0.7, 0.01
r_n = np.random.default_rng(1952)
sds = {}
for N in (10, 100, 1000):
    n_open = int(0.3 * N); rec = []
    for k in range(102000):
        opening = r_n.binomial(N - n_open, a2 * dt)
        closing = ...
        n_open += opening - closing
        if k >= 2000: rec.append(n_open / N)
    sds[N] = ...
print({N: round(float(s), 4) for N, s in sds.items()}, " measured;", {N: round(float(np.sqrt(0.21 / N)), 4) for N in sds}, " theory")

In [ ]:
import numpy as np
def test_sd():
    "for each N the measured sd is within 10% of sqrt(p (1 - p) / N)"
    pinf = 0.3
    for N in (10, 100, 1000):
        assert abs(sds[N] - np.sqrt(pinf * (1 - pinf) / N)) / np.sqrt(pinf * (1 - pinf) / N) < 0.10

test_sd()
print("All checks passed")